In [ ]:
import os
os.chdir("..")

In [ ]:
from jp_qcew import CleanQCEW
import polars as pl

cq = CleanQCEW()

In [ ]:
cq.make_qcew_dataset()

In [ ]:
qcew_dir = self.saving_dir / "qcew"

        for folder_path in qcew_dir.iterdir():
            if not folder_path.is_dir():  # Good practice to ensure it's a folder
                continue
                
            count = 0
            year = str(folder_path)[10:14]  # Double check if folder_path.name is safer here!
            
            for file in folder_path.iterdir():
                year_dir = self.saving_dir / "processed" / "qcew" / str(year)
                year_dir.mkdir(parents=True, exist_ok=True)
                file_path = year_dir / f"data-{count}.parquet"
                
                if not file_path.exists():
                    df = self.clean_txt(file, self.dict_file)
                    # Cast numeric fields
                    df = df.with_columns(
                        pl.col("latitude").cast(pl.Float64, strict=False),
                        pl.col("longitude").cast(pl.Float64, strict=False),
                        pl.col("year").cast(pl.Int64, strict=False),
                        pl.col("qtr").cast(pl.Int64, strict=False),
                        pl.col("first_month_employment").cast(pl.Int64, strict=False),
                        pl.col("second_month_employment").cast(pl.Int64, strict=False),
                        pl.col("third_month_employment").cast(pl.Int64, strict=False),
                        pl.col("total_wages").cast(pl.Int64, strict=False),
                        pl.col("taxable_wages").cast(pl.Int64, strict=False),
                    )
                    # add files id year and qtr

                    df.write_parquet(file_path)
                    print(f"File {file} {count} has been inserted into the database.")
                    count += 1
                else:
                    # Increment count even if skipped so existing files aren't overwritten 
                    # if new files are added later.
                    count += 1 

        # UPDATED RETURN PATH:
        # Uses recursive wildcard (**/) to fetch all data-*.parquet files inside the nested year folders.
        search_path = self.saving_dir / "processed" / "qcew" / "**" / "data-*.parquet"
        
        return self.conn.execute(
            f"SELECT * FROM '{search_path}';"
        ).pl()


In [ ]:
file_path = cq.saving_dir_